## Lesson Overview


In this notebook, you will learn how a simple **natural language processing (NLP)** workflow can turn short pieces of unstructured clinical text into structured information that can be reviewed and analyzed.

You will work with a small, fully synthetic, teaching dataset. The notes are fictional and do not contain protected health information (PHI).

By the end of this activity, you should be able to explain and demonstrate this workflow:

**RAW CLINICAL TEXT → PREPROCESS → NAMED ENTITY RECOGNITION (NER) → NORMALIZE → INTERPRET CONTEXT → STRUCTURED OUTPUT → EVALUATE**

You will practice how to:

- preserve the original text while creating a simple processed version
- use **Python-based NLP methods** to identify medications, substances, conditions, doses, and routes
- normalize different surface forms to a preferred term
- interpret **assertion, temporality, experiencer, and use context**
- create an auditable structured output table
- calculate **precision, recall, and F1** for entity extraction
- distinguish development-set improvement from **held-out evaluation**
- export a small **FAIR-aligned** result with provenance


You do not need to memorize the code in this notebook. The goal is to understand inputs, workflow, and outputs.

Everyone completes the **Foundational Activity**: Steps 0–7, rule improvement, and final evaluation. The **Applied Extension** is an optional continuation that compares a pretrained model with the finalized rule-based system. Both routes finish with the shared FAIR/provenance and wrap-up sections.


## NLP
**Natural language processing (NLP)** uses computational methods to interpret human language and structure it. In this module, NLP helps us identify and structure information found in clinical text.

## Semantics
**Semantics** means meaning. Finding a word is not enough. We also need to understand what that word means in context.

For example:

> **Patient denies heroin use.**

A simple word search finds *heroin*. But the correct interpretation is **not** that the patient currently uses heroin. The word appears in a **negated** statement.

## Why Context Matters

Clinical language can change meaning depending on how a concept is mentioned.

Compare:

- Patient uses fentanyl.
- Patient denies fentanyl use.
- Patient used fentanyl in 2022.
- Father receives methadone.
- Discussed fentanyl overdose risk.

A useful NLP pipeline therefore needs to do more than identify a word. It should preserve enough context to distinguish what is actually being said.

In this notebook we will represent context using four fields:

| Field | Question |
|---|---|
| **Assertion** | Is the concept present, negated, or hypothetical? |
| **Temporality** | Is it current, historical, or future? |
| **Experiencer** | Does it refer to the patient or someone else? |
| **Use context** | Is it treatment, substance use/exposure, education, or another role? |

## Harmonization
**Harmonization** means making data more consistent so it can be interpreted and used together appropriately.

Three types of harmonization are:

1. **Syntactic harmonization** — align representation or format, such as standardizing `Patient_ID`, `PATIENT_ID`, and `patientid` to `patient_id`.
2. **Semantic harmonization** — align meaning, such as mapping *OUD* to *opioid use disorder*.
3. **Measurement / analytic harmonization** — decide whether measurements are similar enough to compare or combine for analysis.

This notebook focuses mainly on **semantic harmonization in unstructured clinical text**.


## How Can NLP Systems Find Information in Text?

There are different ways to build NLP systems.

- **Rule-based NLP** uses rules that people define directly, such as lists of terms, patterns, or context clues. These systems are transparent because we can see exactly why a rule matched.
- **Model-based NLP** uses statistical or machine-learning models that learn patterns from data. Pretrained language models can often recognize relevant language without requiring us to manually list every possible term.

In this module, we will begin with a **simple rule-based NLP pipeline** so you can see how each step works and how errors occur. In the optional **Applied Extension**, you will also compare this approach with a pretrained biomedical NER model.

NOTE: **Neither approach is automatically better.** The best method depends on the task, the available data, and use of the results.


## Step 0: Setup

Run the below cell to import the small set of Python tools we need and create an `outputs` folder for anything you export later.



In [ ]:
# Import the small set of libraries used in this teaching notebook.

# pathlib: creates folders/paths in a way that works across systems
# copy: makes independent copies of rule dictionaries
# json: writes a small metadata file at the end
# re: provides regular expressions for simple text matching
# pandas: stores notes and NLP results in tables
from pathlib import Path
import copy
import json
import re
import pandas as pd

# Make notebook tables easier to read.
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 50)

# Create a folder where the final CSV and metadata files will be saved.
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Record the version of this teaching pipeline.
# Everyone exports the same finalized rule-based pipeline; extension status is recorded separately.
PIPELINE_VERSION = "WISER-M4-teaching-v5"

print("Setup complete.")


Setup complete.


## Step 1: Load and Inspect the Synthetic Notes

Important information is embedded inside sentences in clinical notes. Our goal is to turn selected information in those sentences into a structured dataset (i.e., a table with rows and columns), without losing the original evidence.

Run the next cell to create the synthetic teaching notes.

In [ ]:
# Fully synthetic teaching notes: no real PHI; generic medication names only.
# All development notes may be inspected while building and improving rules.
development_notes = pd.DataFrame({
    "note_id": [
        "M4-01", "M4-02", "M4-03", "M4-04", "M4-05",
        "M4-06", "M4-07", "M4-08", "M4-09", "M4-10", "M4-11"
    ],
    "text": [
        "Patient denies heroin use. Continues buprenorphine for opioid\nuse disorder.",
        "History of fentanyl use in 2022; no current opioid use reported.",
        "Father receives methadone for OUD.\tPatient reports no opioid use.",
        "Naloxone 4 mg\nintranasal kit offered for overdose education.",
        "Patient reports taking buprenorphine as prescribed.",
        "Urine toxicology was negative for fentanyl.",
        "Patient says naloxone is kept at home.",
        "Considering starting buprenorphine\tnext week.",
        "Patient reports taking bup as prescribed.",
        "Patient reports fent use was years ago.",
        "Patient denies cravings but reports fentanyl use yesterday."
    ]
})

development_notes["split"] = "development"

print("Development notes:")
display(development_notes[["note_id", "text"]])


Development notes:


,note_id,text
0,M4-01,Patient denies heroin use. Continues buprenorphine for opioid\nuse disorder.
1,M4-02,History of fentanyl use in 2022; no current opioid use reported.
2,M4-03,Father receives methadone for OUD.\tPatient reports no opioid use.
3,M4-04,Naloxone 4 mg\nintranasal kit offered for overdose education.
4,M4-05,Patient reports taking buprenorphine as prescribed.
5,M4-06,Urine toxicology was negative for fentanyl.
6,M4-07,Patient says naloxone is kept at home.
7,M4-08,Considering starting buprenorphine\tnext week.
8,M4-09,Patient reports taking bup as prescribed.
9,M4-10,Patient reports fent use was years ago.


### Stop and Notice

Before running any NLP code, read the notes.

Ask yourself:

1. Which concepts would be useful to extract?
2. Which mentions should **not** be interpreted as current patient substance use?
3. Which words are different surface forms of the same or related concepts?

This manual reading is important. NLP output should always be interpreted against the source text.

## Step 2: Preprocessing of Text Data

**Preprocessing** prepares text for analysis. Depending on the text and the NLP task, preprocessing may include:

- **Formatting cleanup** — replace tabs, line breaks, or extra spacing that interrupt phrases.
- **Case standardization** — convert text to a consistent case when case-sensitive matching is a problem.
- **Sentence segmentation** — identify where sentences begin and end.
- **Tokenization** — split text into words, punctuation, or other units used by the NLP method.
- **De-identification** — remove or mask protected health information when working with identifiable clinical text.
- **Encoding cleanup** — standardize unusual symbols, quotation marks, or text-encoding artifacts.

> Not every dataset needs every preprocessing step. The goal is to fix problems that could interfere with the NLP task without unnecessarily changing the text.

For this beginner workflow, we will keep preprocessing intentionally simple:

- preserve the original text data
- create a lowercase copy for matching
- replace line breaks (`\n`) and tabs (`\t`) with a single space

We replace each line break or tab with exactly **one space**. For these ASCII teaching notes, lowercasing also preserves length. This keeps the text the same length, so the `start` and `end` character positions in the string detected by NER can be traced back to the same positions in the original note.

We will **not** remove all punctuation or numbers because they may contain useful information such as dose or sentence boundaries.

> This is a **syntactic harmonization** step: we make representation more consistent without changing the source or meaning.

### Why keep both the raw and processed text?

You will see both `text` and `text_processed` in the notebook.

- `text` is the original source text.
- `text_processed` shows a simple cleaned version.

This offset guarantee is specific to these notes: some Unicode characters change length when lowercased, so other datasets need an offset-preservation check.


In [ ]:
def preprocess_text(text):
    """
    Create a simple processed copy of a clinical note for this NLP activity.

    The function:
    - converts text to lowercase for case-insensitive matching
    - replaces each newline and tab with exactly one space

    Replacing one character with one character preserves string length in
    these synthetic notes, so NER character offsets remain aligned with
    the original note.

    Parameters
    ----------
    text : str
        Original clinical-note text.

    Returns
    -------
    str
        Processed text used by the NLP pipeline.
    """
    # Standardize capitalization for matching.
    text = str(text).lower()

    # Replace simple formatting artifacts without changing string length.
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")

    return text


# Create a processed-text column while preserving the original text.
development_notes["text_processed"] = development_notes["text"].map(preprocess_text)

# Compare the original and processed development notes.
development_notes[["note_id", "text", "text_processed"]]


,note_id,text,text_processed
0,M4-01,Patient denies heroin use. Continues buprenorphine for opioid\nuse disorder.,patient denies heroin use. continues buprenorphine for opioid use disorder.
1,M4-02,History of fentanyl use in 2022; no current opioid use reported.,history of fentanyl use in 2022; no current opioid use reported.
2,M4-03,Father receives methadone for OUD.\tPatient reports no opioid use.,father receives methadone for oud. patient reports no opioid use.
3,M4-04,Naloxone 4 mg\nintranasal kit offered for overdose education.,naloxone 4 mg intranasal kit offered for overdose education.
4,M4-05,Patient reports taking buprenorphine as prescribed.,patient reports taking buprenorphine as prescribed.
5,M4-06,Urine toxicology was negative for fentanyl.,urine toxicology was negative for fentanyl.
6,M4-07,Patient says naloxone is kept at home.,patient says naloxone is kept at home.
7,M4-08,Considering starting buprenorphine\tnext week.,considering starting buprenorphine next week.
8,M4-09,Patient reports taking bup as prescribed.,patient reports taking bup as prescribed.
9,M4-10,Patient reports fent use was years ago.,patient reports fent use was years ago.


## Step 3: Named Entity Recognition (NER)

**Named entity recognition (NER)** identifies important spans of text and assigns them a type.

Our small rule-based information extraction system will identify mentions of:

- **MEDICATION**
- **SUBSTANCE**
- **CONDITION**
- **DOSE**
- **ROUTE**

A production NLP system could use much more sophisticated models. We are starting with rules because they are transparent: you can see exactly why the system found or missed a term.

> **Important:** NER only tells us that something was mentioned. It does **not** yet tell us anything about assertion (the concept being present, negated, or hypothetical), temporality (the concept being current, historical, or future), the experiencer (patient or someone else), or use context (treatment, substance use, education, etc.).
In the next table, `mention` is the matched text and `entity_type` is its label. `start` counts characters from 0; `end` is the first character after the mention. Thus `text[start:end]` recovers the span. Labels here are task-specific: fentanyl is categorized as `SUBSTANCE` for this activity even though it also has medical uses.


In [ ]:
# Define the terms our starting rule-based extractor can recognize.
# Each dictionary key is an entity type, and each list contains terms to find.
BASE_TERM_PATTERNS = {
    "MEDICATION": ["naloxone", "buprenorphine", "methadone", "oxycodone"],
    "SUBSTANCE": ["heroin", "fentanyl", "opioid"],
    "CONDITION": ["opioid use disorder", "oud"],
}

# Simple patterns for medication-related attributes.
DOSE_PATTERN = r"\b\d+(?:\.\d+)?\s?(?:mg|mcg|g)\b"
ROUTE_TERMS = ["intranasal", "sublingual", "oral", "intravenous"]


def extract_entities(text, term_patterns=None):
    """
    Find predefined clinical entities and medication-related attributes.

    The function searches for:
    - dictionary terms in `term_patterns`
    - simple dose expressions such as "4 mg"
    - route terms such as "intranasal"

    If candidate spans overlap, the longer span is kept. For example,
    "opioid use disorder" is preferred over the shorter word "opioid"
    when both begin within the same phrase.

    Parameters
    ----------
    text : str
        Text to search.

    term_patterns : dict or None
        Mapping from entity types to lists of terms. If None,
        BASE_TERM_PATTERNS is used.

    Returns
    -------
    list of dict
        One dictionary per extracted entity with:
        `mention`, `start`, `end`, and `entity_type`.
    """
    if term_patterns is None:
        term_patterns = BASE_TERM_PATTERNS

    # Collect all candidate matches before resolving overlaps.
    candidates = []

    # Use a lowercase copy for case-insensitive matching.
    # Lowercasing these synthetic notes does not change character positions.
    lower = str(text).lower()

    # 1) Match dictionary terms such as "heroin", "naloxone", or "OUD".
    for entity_type, terms in term_patterns.items():
        for term in sorted(terms, key=len, reverse=True):
            for match in re.finditer(rf"\b{re.escape(term)}\b", lower):
                candidates.append({
                    "mention": text[match.start():match.end()],
                    "start": match.start(),
                    "end": match.end(),
                    "entity_type": entity_type
                })

    # 2) Match simple dose expressions such as "4 mg".
    for match in re.finditer(DOSE_PATTERN, lower):
        candidates.append({
            "mention": text[match.start():match.end()],
            "start": match.start(),
            "end": match.end(),
            "entity_type": "DOSE"
        })

    # 3) Match route terms such as "intranasal".
    for route in ROUTE_TERMS:
        for match in re.finditer(rf"\b{re.escape(route)}\b", lower):
            candidates.append({
                "mention": text[match.start():match.end()],
                "start": match.start(),
                "end": match.end(),
                "entity_type": "ROUTE"
            })

    # Prefer the longest span globally, then the earliest start for ties.
    candidates = sorted(
        candidates,
        key=lambda x: (-(x["end"] - x["start"]), x["start"])
    )

    # Keep only non-overlapping candidates.
    kept = []
    for candidate in candidates:
        overlaps = any(
            not (
                candidate["end"] <= existing["start"]
                or candidate["start"] >= existing["end"]
            )
            for existing in kept
        )
        if not overlaps:
            kept.append(candidate)

    return sorted(kept, key=lambda x: x["start"])


# Example: run NER on the first processed development note.
example_text = development_notes.loc[0, "text_processed"]

print("Example note:")
print(example_text)

print("\nEntities found:")
display(pd.DataFrame(extract_entities(example_text)))


Example note:
patient denies heroin use. continues buprenorphine for opioid use disorder.

Entities found:


,mention,start,end,entity_type
0,heroin,15,21,SUBSTANCE
1,buprenorphine,37,50,MEDICATION
2,opioid use disorder,55,74,CONDITION


### Assess the Output of NER



1.   Are all relevant entity mentions found?
2.   Are the mentioned entities categorized correctly?
3. Does the named entity recognition give information about the context?




### NOTE: Importance of Preprocessing

Before preprocessing, the line break interrupts the phrase **“opioid use disorder,”** so the rule-based extractor only finds **“opioid”** and labels it as a **SUBSTANCE**.

After preprocessing replaces the line break with a space, the full phrase is recognized correctly as a **CONDITION**. This shows how small formatting artifacts can change NLP results.

In [ ]:
raw_text = development_notes.loc[0, "text"]
clean_text = development_notes.loc[0, "text_processed"]

print("\nBEFORE PREPROCESSING:")
display(pd.DataFrame(extract_entities(raw_text)))

print("\n\nAFTER PREPROCESSING:")
display(pd.DataFrame(extract_entities(clean_text)))


BEFORE PREPROCESSING:


,mention,start,end,entity_type
0,heroin,15,21,SUBSTANCE
1,buprenorphine,37,50,MEDICATION
2,opioid,55,61,SUBSTANCE




AFTER PREPROCESSING:


,mention,start,end,entity_type
0,heroin,15,21,SUBSTANCE
1,buprenorphine,37,50,MEDICATION
2,opioid use disorder,55,74,CONDITION


### What Should You See?

For the first note, the baseline should identify:

- **heroin** as a substance
- **buprenorphine** as a medication
- **opioid use disorder** as a condition

Notice what NER has **not** done yet: it has not represented that heroin use is denied.

That is why context interpretation comes later.

## Step 4: Normalization

After NER identifies a term, we may still need to **standardize how that term is represented**.

Different abbreviations, shorthand forms, or chemical forms can refer to the same underlying concept.

**Normalization** maps these different expressions to a consistent preferred term.

Examples:

- **OUD → opioid use disorder**
- **naloxone hydrochloride → naloxone**
- **oxy → oxycodone**

In this synthetic teaching activity, **oxy** is treated as shorthand for **oxycodone**.

Normalization helps make structured output more consistent. For example, `OUD` and `opioid use disorder` can both be represented using the preferred term **opioid use disorder**.

This is **semantic harmonization** because different expressions are being aligned to the same underlying meaning. Lowercasing alone is case standardization during preprocessing; it is not the semantic-normalization step.

These mappings depend on the task: the salt form is collapsed to the ingredient for this activity, losing formulation detail. Abbreviations such as `oxy` can be ambiguous outside these examples. The next cell demonstrates normalization directly; adding a normalization mapping alone does not make NER recognize a new term.


In [ ]:
# This dictionary performs simple semantic normalization.
# The LEFT side is a surface form that may appear in text.
# The RIGHT side is the preferred term used in our structured output.
BASE_NORMALIZATION_MAP = {
    "naloxone": "naloxone",
    "buprenorphine": "buprenorphine",
    "methadone": "methadone",
    "oxycodone": "oxycodone",
    "oud": "opioid use disorder",
    "opioid use disorder": "opioid use disorder",
    "heroin": "heroin",
    "fentanyl": "fentanyl",
    "opioid": "opioid",
    "oxy": "oxycodone",
    "naloxone hydrochloride": "naloxone",
}


def normalize_mention(mention, normalization_map=None):
    """
    Map an extracted term to a preferred term using a normalization dictionary.

    The lookup is case-insensitive. If the term is not in the dictionary,
    the function returns the original mention unchanged rather than treating
    lowercasing alone as semantic normalization.

    Parameters
    ----------
    mention : str
        Extracted text mention.

    normalization_map : dict or None
        Mapping from surface forms to preferred terms. If None,
        BASE_NORMALIZATION_MAP is used.

    Returns
    -------
    str
        Preferred normalized term, or the original mention if no mapping exists.
    """
    if normalization_map is None:
        normalization_map = BASE_NORMALIZATION_MAP

    lookup_key = str(mention).lower()
    return normalization_map.get(lookup_key, mention)


# Show examples in which semantic normalization changes the representation.
print("Examples of semantic normalization:\n")

for term in ["OUD", "naloxone hydrochloride", "oxy"]:
    normalized = normalize_mention(term)
    print(f"{term} -> {normalized}")


Examples of semantic normalization:

OUD -> opioid use disorder
naloxone hydrochloride -> naloxone
oxy -> oxycodone


### Optional: Viewing Normalization as a Crosswalk

A **crosswalk** is a table that shows how terms or variables from one representation map to another.

The normalization dictionary used above can also be represented as a simple terminology crosswalk:

| Original term | Preferred term |
|---|---|
| OUD | opioid use disorder |
| naloxone hydrochloride | naloxone |
| oxy | oxycodone |


This is another way to document the semantic harmonization performed during normalization.

### Optional: What is Concept Linking?

A normalized term is still a text label. In real research, we may go one step further and connect it to a standardized concept in an established terminology, usually with a stable identifier.

**RxNorm** is a standardized clinical drug terminology maintained by the U.S. National Library of Medicine (NLM). It provides consistent names and identifiers for medication concepts. Each RxNorm concept has a unique identifier called an **RxCUI (RxNorm Concept Unique Identifier)**.

If a clinical note or medication record contains enough detail to identify a specific drug concept, a production system could link that information to an RxCUI. For example:

| Standardized medication concept | Terminology | RxCUI |
|---|---|---:|
| naloxone HCl 4 mg per 0.1 mL nasal spray | RxNorm | **1725059** |
| buprenorphine 2 mg sublingual tablet | RxNorm | **351264** |

These examples show why **concept linking can be more specific than normalization**. Normalization may map a mention to a preferred ingredient name such as **naloxone**, while an RxNorm concept can represent a more specific medication concept that includes information such as ingredient, strength, and dose form.

So:

- **Normalization** maps different expressions to a preferred term.
- **Concept linking** connects that term to a standardized concept and identifier.

> In this activity, we perform normalization but do not query RxNorm or assign RxCUIs in code.


## Step 5: Interpret Context

NER tells us what terms were mentioned, but not **what those mentions mean in the sentence**.

For each extracted mention, we will describe four dimensions of context:

1. **Assertion** — Is the concept affirmed, negated, or hypothetical?
2. **Temporality** — Is it current, historical, or future?
3. **Experiencer** — Does it refer to the patient or someone else?
4. **Use context** — What role does the concept play in the sentence?

---

### 1. Assertion

**Assertion** describes whether the mention is being affirmed, denied, or discussed as a possibility.

Possible labels:

- `PRESENT` — the concept is affirmed
- `NEGATED` — the concept is denied
- `HYPOTHETICAL` — the concept is being considered or discussed as a possibility

Example:

> Patient denies heroin use.

`heroin → assertion = NEGATED`

> **Important:** `PRESENT` does not necessarily mean that the patient took or used something. It means only that the mention is **affirmed rather than negated or hypothetical**.

For example, a naloxone kit may be *offered* or *discussed* and still have `assertion = PRESENT`.

---

### 2. Temporality

**Temporality** describes when the mentioned concept applies.

Possible labels:

- `CURRENT` — current or recent
- `HISTORICAL` — clearly in the past
- `FUTURE` — expected or planned for the future

Example:

> Patient reports fentanyl use in 2022.

`fentanyl → temporality = HISTORICAL`

In this simplified teaching activity, `CURRENT` includes the **current or recent clinical episode**. Clearly remote references such as *“in 2022”* or *“years ago”* are labeled `HISTORICAL`.

---

### 3. Experiencer

**Experiencer** identifies who the mention refers to.

Possible labels:

- `PATIENT` — the concept refers to the patient
- `FAMILY` — the concept refers to a family member

Example:

> Father receives methadone for OUD.

`methadone → experiencer = FAMILY`

This distinction matters because simply finding a medication, substance, or condition in a note does not mean that it applies to the patient.

---

### 4. Use Context

**Use context** describes the role that the extracted concept plays in the sentence.

Possible labels in this activity include:

- `TREATMENT` — medication mentioned in treatment-related wording
- `SUBSTANCE_USE_OR_EXPOSURE` — substance-related mention; read assertion before inferring use
- `EDUCATION` — discussion, counseling, or risk education
- `CONDITION` — a condition mention
- `ATTRIBUTE` — a dose or route
- `OTHER` — no listed use-context rule applies

Example:

> Continues buprenorphine for OUD.

`buprenorphine → use_context = TREATMENT`

#### Entity Type vs. Use Context

**Entity type** describes **what the extracted term is**.

For example:

`buprenorphine → entity_type = MEDICATION`

**Use context** describes **what role that term plays in the sentence**.

For example:

`buprenorphine → use_context = TREATMENT`

The same entity type can therefore appear in different contexts depending on what the sentence says.

#### Dose and Route

In this activity, use context of `DOSE` and `ROUTE` mentions is `ATTRIBUTE`.

For example:

> Naloxone 4 mg intranasal kit offered for overdose education.

- `4 mg → entity_type = DOSE; use_context = ATTRIBUTE`
- `intranasal → entity_type = ROUTE; use_context = ATTRIBUTE`

A more advanced NLP pipeline could also determine **which medication a dose or route belongs to**. This is often handled through **relation extraction**, which is outside the scope of this beginner activity.

---

**NOTE:** These context rules are intentionally simple. Their purpose is to demonstrate how context changes the interpretation of extracted terms, not to reproduce a production clinical NLP system.

The rules default to `PRESENT`, `CURRENT`, and `PATIENT` when no competing cue is found; these are heuristic assignments, not verified facts. In M4-06, `NEGATED` describes a negative toxicology finding, which alone does not establish that no substance use occurred.


In [ ]:
def sentence_for_span(text, start, end):
    """
    Return the sentence-like section of text that contains an extracted entity.

    Parameters
    ----------
    text : str
        The full clinical note.
    start : int
        Starting character position of the entity.
    end : int
        Ending character position of the entity.

    Returns
    -------
    str
        The sentence containing the entity.

    Note
    ----
    For this teaching example, periods and semicolons are treated as
    sentence boundaries.
    """

    # Find the closest period or semicolon BEFORE the entity.
    left = max(
        text.rfind(".", 0, start),
        text.rfind(";", 0, start)
    )

    # Find the closest period or semicolon AFTER the entity.
    right_candidates = [
        p for p in [
            text.find(".", end),
            text.find(";", end)
        ]
        if p != -1
    ]

    # If no later sentence boundary is found, use the end of the note.
    right = min(right_candidates) if right_candidates else len(text)

    # Return only the sentence-like section containing the entity.
    return text[left + 1:right].strip()


def interpret_context(text, entity, normalized_concept):
    """
    Interpret four kinds of context for one extracted and normalized entity.

    Parameters
    ----------
    text : str
        The full clinical note.
    entity : dict
        The extracted entity, including its text span and entity type.
    normalized_concept : str
        The preferred term produced by the normalization step.

    Returns
    -------
    dict
        Context labels for:
        - assertion
        - temporality
        - experiencer
        - use_context

    Note
    ----
    The normalized concept is passed alongside the extracted entity so the
    pipeline keeps the standardized representation available at this stage.
    The simple context rules below currently rely on sentence cues and
    entity type rather than the normalized value itself.

    These are simplified teaching rules rather than a production
    clinical NLP context model.
    """

    # Get only the sentence containing this entity.
    # Limiting the rules to one sentence helps prevent context cues
    # in another sentence from affecting this entity.
    sentence = sentence_for_span(
        text,
        entity["start"],
        entity["end"]
    )

    # Convert the sentence to lowercase for simple rule matching.
    sent_lower = sentence.lower()

    # ---------------------------------------------------------
    # 1) ASSERTION
    # ---------------------------------------------------------
    # Assertion asks:
    # Is the concept affirmed, negated, or hypothetical?

    negation_cues = [
        "denies",
        "denied",
        "no ",
        "negative for",
        "negative"
    ]

    hypothetical_cues = [
        "considering",
        "may start",
        "plan to",
        "planned",
        "next week"
    ]

    # If a negation cue appears anywhere in the entity's sentence,
    # label the mention as NEGATED.
    #
    # This is intentionally simple. A sentence containing multiple
    # ideas can still cause this rule to make a context error.
    if any(cue in sent_lower for cue in negation_cues):
        assertion = "NEGATED"

    # If the sentence describes something being considered or planned,
    # label the mention as HYPOTHETICAL.
    elif any(cue in sent_lower for cue in hypothetical_cues):
        assertion = "HYPOTHETICAL"

    # Otherwise, treat the mention as affirmed.
    else:
        assertion = "PRESENT"

    # ---------------------------------------------------------
    # 2) TEMPORALITY
    # ---------------------------------------------------------
    # Temporality asks:
    # Does the mention refer to the past, present/recent, or future?

    historical_cues = [
        "history of",
        "in 2022",
        "years ago",
        "previously"
    ]

    future_cues = [
        "next week",
        "considering",
        "plan to",
        "planned"
    ]

    if any(cue in sent_lower for cue in future_cues):
        temporality = "FUTURE"

    elif any(cue in sent_lower for cue in historical_cues):
        temporality = "HISTORICAL"

    else:
        temporality = "CURRENT"

    # ---------------------------------------------------------
    # 3) EXPERIENCER
    # ---------------------------------------------------------
    # Experiencer asks:
    # Who does the extracted concept refer to?

    family_cues = [
        "father",
        "mother",
        "sister",
        "brother",
        "parent"
    ]

    if any(cue in sent_lower for cue in family_cues):
        experiencer = "FAMILY"
    else:
        experiencer = "PATIENT"

    # ---------------------------------------------------------
    # 4) USE CONTEXT
    # ---------------------------------------------------------
    # Use context asks:
    # What role does the extracted concept play in the sentence?

    # Dose and route are treated as medication-related attributes.
    if entity["entity_type"] in {"DOSE", "ROUTE"}:
        use_context = "ATTRIBUTE"

    # Mentions related to counseling or overdose education.
    elif (
        "education" in sent_lower
        or "discussed" in sent_lower
        or "overdose risk" in sent_lower
    ):
        use_context = "EDUCATION"

    # Medication mentions associated with treatment-related wording.
    elif entity["entity_type"] == "MEDICATION" and (
        "continues" in sent_lower
        or "taking" in sent_lower
        or "receives" in sent_lower
        or "starting" in sent_lower
        or "offered" in sent_lower
        or "prescribed" in sent_lower
    ):
        use_context = "TREATMENT"

    # Substance mentions are categorized as substance use or exposure.
    elif entity["entity_type"] == "SUBSTANCE":
        use_context = "SUBSTANCE_USE_OR_EXPOSURE"

    # Condition mentions are categorized as conditions.
    elif entity["entity_type"] == "CONDITION":
        use_context = "CONDITION"

    # Anything not covered by the teaching rules goes into OTHER.
    else:
        use_context = "OTHER"

    # Return all four context labels together.
    return {
        "assertion": assertion,
        "temporality": temporality,
        "experiencer": experiencer,
        "use_context": use_context,
    }


# ---------------------------------------------------------
# EXAMPLE: Normalize and interpret ALL entities in the first note
# ---------------------------------------------------------

example_text = development_notes.loc[0, "text_processed"]

# Step 1: Use NER to find all entities in the note.
example_entities = extract_entities(example_text)

# Create an empty list where we will store the results.
example_results = []

# Step 2: Process each extracted entity.
for entity in example_entities:

    # Normalize the extracted mention to its preferred term.
    normalized = normalize_mention(entity["mention"])

    # Interpret the context of the entity.
    # We pass both the extracted entity and its normalized concept.
    context = interpret_context(
        example_text,
        entity,
        normalized
    )

    # Store the NER result, normalized concept, and context labels together.
    example_results.append({
        "mention": entity["mention"],
        "entity_type": entity["entity_type"],
        "normalized_concept": normalized,
        **context
    })

# Step 3: Convert the results into a structured table.
example_results_df = pd.DataFrame(example_results)

print("Clinical note:")
print(example_text)

print("\nNER, normalization, and context interpretation:")

display(example_results_df)


Clinical note:
patient denies heroin use. continues buprenorphine for opioid use disorder.

NER, normalization, and context interpretation:


,mention,entity_type,normalized_concept,assertion,temporality,experiencer,use_context
0,heroin,SUBSTANCE,heroin,NEGATED,CURRENT,PATIENT,SUBSTANCE_USE_OR_EXPOSURE
1,buprenorphine,MEDICATION,buprenorphine,PRESENT,CURRENT,PATIENT,TREATMENT
2,opioid use disorder,CONDITION,opioid use disorder,PRESENT,CURRENT,PATIENT,CONDITION


### Check the Meaning, Not Just the Labels

For the first note:

> **Patient denies heroin use. Continues buprenorphine for opioid use disorder.**

the correct structured interpretation is:

- **heroin** → patient + current + **NEGATED** + substance use/exposure
- **buprenorphine** → patient + current + **PRESENT** + treatment
- **opioid use disorder** → patient + current + **PRESENT** + condition

Only **heroin** is negated. The word *denies* appears in the first sentence and should not affect entities in the next sentence. The `sentence_for_span()` helper therefore limits context checking to the sentence containing each entity.

> **Note:** These rules are intentionally simple and may not interpret every sentence correctly. Later, you will inspect the pipeline output and identify a context error.


## Step 6: Run the Full Pipeline

Now we will combine the earlier steps into one reusable function.

The main structured output will keep both:

- **source evidence**: `note_id`, `mention`, `start`, `end`
- **NLP interpretation**: `entity_type`, `normalized_concept`, the four context fields, and `data_split`

Keeping the source evidence makes the output **auditable**: another person can trace each structured row back to the exact mention in the note.

The pipeline version will be recorded later in the **FAIR/provenance metadata** rather than repeated in every row of the main student-facing table.

`mention` is taken from `text_processed`; the offsets also locate the corresponding raw span in these notes, whose capitalization or whitespace may differ. `data_split` records the dataset label; all notes used so far are `development`. Preprocessing was already completed in Step 2; the function below combines Steps 3–5.


In [ ]:
def run_pipeline(
    notes_df,
    term_patterns=None,
    normalization_map=None
):
    """
    Run the complete teaching NLP pipeline on a table of clinical notes.

    For each note, the function:
    1. identifies entities and their character offsets
    2. normalizes each extracted mention to a preferred term
    3. interprets assertion, temporality, experiencer, and use context
    4. stores the results in a structured table

    Parameters
    ----------
    notes_df : pandas.DataFrame
        Table containing at least `note_id` and `text_processed`.
        A `split` column may also be included.

    term_patterns : dict or None
        Entity rules used by `extract_entities()`. If None,
        BASE_TERM_PATTERNS is used.

    normalization_map : dict or None
        Surface-form mappings used by `normalize_mention()`. If None,
        BASE_NORMALIZATION_MAP is used.

    Returns
    -------
    pandas.DataFrame
        One row per extracted entity with source evidence, entity type,
        normalized term, context labels, and data split.
    """
    # Collect one structured row for every entity found across all notes.
    all_rows = []

    # Process one note at a time.
    for _, note in notes_df.iterrows():

        # Step 1: identify entity mentions and their character offsets.
        entities = extract_entities(
            note["text_processed"],
            term_patterns=term_patterns
        )

        # Step 2: normalize and interpret each extracted entity.
        for entity in entities:
            normalized = normalize_mention(
                entity["mention"],
                normalization_map=normalization_map
            )

            context = interpret_context(
                note["text_processed"],
                entity,
                normalized
            )

            # Step 3: combine source evidence and NLP interpretation.
            all_rows.append({
                "note_id": note["note_id"],
                "data_split": note.get("split", "unspecified"),
                "mention": entity["mention"],
                "start": entity["start"],
                "end": entity["end"],
                "entity_type": entity["entity_type"],
                "normalized_concept": normalized,
                **context,
            })

    # Preserve the output columns even when no entities are found.
    columns = [
        "note_id", "data_split", "mention", "start", "end", "entity_type",
        "normalized_concept", "assertion", "temporality", "experiencer", "use_context",
    ]
    return pd.DataFrame(all_rows, columns=columns)


# ---------------------------------------------------------
# EXAMPLE: RUN THE BASELINE PIPELINE
# ---------------------------------------------------------

# Run the baseline pipeline on the development notes.
# These are the notes we may inspect while deciding how to improve the rules.
baseline_entities = run_pipeline(
    development_notes,
    normalization_map=BASE_NORMALIZATION_MAP
)

display(baseline_entities)


,note_id,data_split,mention,start,end,entity_type,normalized_concept,assertion,temporality,experiencer,use_context
0,M4-01,development,heroin,15,21,SUBSTANCE,heroin,NEGATED,CURRENT,PATIENT,SUBSTANCE_USE_OR_EXPOSURE
1,M4-01,development,buprenorphine,37,50,MEDICATION,buprenorphine,PRESENT,CURRENT,PATIENT,TREATMENT
2,M4-01,development,opioid use disorder,55,74,CONDITION,opioid use disorder,PRESENT,CURRENT,PATIENT,CONDITION
3,M4-02,development,fentanyl,11,19,SUBSTANCE,fentanyl,PRESENT,HISTORICAL,PATIENT,SUBSTANCE_USE_OR_EXPOSURE
4,M4-02,development,opioid,44,50,SUBSTANCE,opioid,NEGATED,CURRENT,PATIENT,SUBSTANCE_USE_OR_EXPOSURE
5,M4-03,development,methadone,16,25,MEDICATION,methadone,PRESENT,CURRENT,FAMILY,TREATMENT
6,M4-03,development,oud,30,33,CONDITION,opioid use disorder,PRESENT,CURRENT,FAMILY,CONDITION
7,M4-03,development,opioid,54,60,SUBSTANCE,opioid,NEGATED,CURRENT,PATIENT,SUBSTANCE_USE_OR_EXPOSURE
8,M4-04,development,naloxone,0,8,MEDICATION,naloxone,PRESENT,CURRENT,PATIENT,EDUCATION
9,M4-04,development,4 mg,9,13,DOSE,4 mg,PRESENT,CURRENT,PATIENT,ATTRIBUTE


## This Is the Main Output of the NLP Workflow

Each row represents **one entity mention extracted from the clinical text**.

Notice that the pipeline has transformed information that was originally embedded in sentences into a structured table. For each mention, we can now see:

- **what text was found** (`mention`)
- **where it appeared in the note** (`start`, `end`)
- **what type of entity it is** (`entity_type`)
- **what preferred term it was normalized to** (`normalized_concept`)
- **whether the mention was affirmed, negated, or hypothetical** (`assertion`)
- **whether it refers to the current, past, or future context** (`temporality`)
- **who the mention refers to** (`experiencer`)
- **what role the mention plays in the sentence** (`use_context`)

### Why is this useful?

A simple keyword search might tell us that a note contains the word **fentanyl**. This structured output gives us much more information.

For example, it can distinguish between:

- a patient **currently reporting fentanyl use**
- a patient **denying fentanyl use**
- **historical** fentanyl use
- fentanyl mentioned only during **overdose education**
- fentanyl use referring to **someone other than the patient**

This makes the information easier to review, analyze, compare across notes, and check against the original text.

**MAIN IDEA:** The goal of the NLP pipeline is not just to find words. It is to convert relevant information in unstructured text into structured data while preserving enough context to interpret that information correctly.

## Step 7: Did the NLP Work?

We should not assume that the pipeline is correct just because it produced a table. We compare its predictions with **reference annotations**: human-defined correct answers describing the text spans and entity types that should be extracted.

### Development notes and reference annotations

All **11 development notes** may be inspected while building and improving the rules. For quantitative scoring, we use a **5-note subset** with complete reference annotations. `development_reference_ids` lists those notes, `development_scoring_notes` holds their text, and `gold_development` holds their reference entities.

**The reference annotations are not a third dataset.** They are the correct answers attached to the scored development notes. They describe what is present in the text, whether or not the baseline recognizes it. Predictions from the other six development notes are unscored, not false positives.

Our evaluation sequence is:

**baseline rules → score the annotated development subset → inspect errors → improve rules → re-score development → freeze rules → one final held-out evaluation**

We use development errors to choose changes. Later, a separate evaluation checks the finalized rules. Even without training a machine-learning model, hand-written rules can become tailored to examples used during development. These tiny synthetic examples demonstrate the evaluation process, not real-world performance.

### What counts as a correct entity?

A prediction must match the reference on **note ID, exact start and end offsets, and entity type**. This is **exact-span + entity-type matching**. Offsets distinguish repeated occurrences of the same word.

- **True positive (TP):** an exactly matched prediction.
- **False positive (FP):** a prediction without an exact reference match in a scored note.
- **False negative (FN):** a reference entity without an exact prediction match.

A wrong span or type produces both an FP and an FN.

| Metric | Question | Calculation |
|---|---|---|
| Precision | Of the entities predicted, how many were correct? | TP / (TP + FP) |
| Recall | Of the reference entities, how many were found? | TP / (TP + FN) |
| F1 | How well does the system balance precision and recall? | 2 × precision × recall / (precision + recall) |

The code reports 0 when a denominator is zero. For example, if 8 of 10 predictions are correct and the reference contains 12 entities, precision = **0.80**, recall ≈ **0.67**, and F1 ≈ **0.73**.

These metrics assess **entity extraction only**, not normalization or the four context fields. We inspect those separately.

### Create the development reference annotations

Every target mention in each scoring note is annotated, including negated, historical, and family mentions. Use the full `opioid use disorder` phrase rather than a nested `opioid` span; annotate doses and routes separately. Words such as cravings and overdose are outside this activity's chosen target inventory.

`make_reference()` locates a span already selected by a human; it does not use the NLP extractor to decide what is correct. `occurrence=0` means the first matching occurrence. Run the next cell to create and inspect `gold_development`.


In [ ]:
# ---------------------------------------------------------
# CREATE HUMAN-DEFINED REFERENCE ANNOTATIONS
# ---------------------------------------------------------
#
# To evaluate NER, we need correct answers to compare against
# the entities produced by the NLP system.
#
# These correct answers are called REFERENCE ANNOTATIONS
# (sometimes called "gold-standard" annotations).
#
# A human decides:
# - which text spans should count as entities
# - what entity type each span should receive
#
# The helper function below does NOT decide what is correct.
# It only finds the character positions of an entity that
# has already been chosen for the reference set.


def make_reference(note_df, note_id, mention, entity_type, occurrence=0):
    """
    Create one reference annotation for a known entity mention.

    Parameters
    ----------
    note_df : pandas.DataFrame
        Table containing the clinical notes.

    note_id : str
        ID of the note containing the entity.

    mention : str
        The entity text that should be located.

    entity_type : str
        The correct entity label, such as MEDICATION,
        SUBSTANCE, CONDITION, DOSE, or ROUTE.

    occurrence : int
        Which occurrence to use if the same term appears more
        than once in the note. The default is the first occurrence.

    Returns
    -------
    tuple
        note_id, matched text, start position, end position,
        and correct entity type.

    Note
    ----
    The entity is located in text_processed because that is the
    version of the text analyzed by the NER system.

    Our preprocessing replaces each line break or tab with one
    space, so the length of the text and its character positions
    remain unchanged.
    """

    # Select the requested note and check that it exists.
    note_rows = note_df.loc[
        note_df["note_id"] == note_id,
        "text_processed"
    ]

    if len(note_rows) != 1:
        raise ValueError(f"Expected exactly one note for note_id {note_id!r}.")

    text = note_rows.iloc[0]

    # Find every occurrence of the chosen mention.
    # IGNORECASE allows capitalization differences.
    matches = list(
        re.finditer(
            re.escape(mention),
            text,
            flags=re.IGNORECASE
        )
    )

    # Stop with a clear error if the requested mention cannot be found.
    if occurrence < 0 or occurrence >= len(matches):
        raise ValueError(
            f"Could not find occurrence {occurrence} "
            f"of {mention!r} in {note_id}."
        )

    # Select the requested occurrence.
    match = matches[occurrence]

    # Return the complete reference annotation.
    return (
        note_id,
        text[match.start():match.end()],
        match.start(),
        match.end(),
        entity_type
    )


# ---------------------------------------------------------
# DEVELOPMENT SCORING SUBSET
# ---------------------------------------------------------
#
# All 11 development notes can be inspected while developing
# the NLP pipeline.
#
# For the quantitative NER teaching check, we use a smaller
# subset of 5 notes with complete reference annotations.

development_reference_ids = [
    "M4-01",
    "M4-03",
    "M4-04",
    "M4-09",
    "M4-10"
]

development_scoring_notes = development_notes[
    development_notes["note_id"].isin(development_reference_ids)
].copy()


# ---------------------------------------------------------
# DEVELOPMENT REFERENCE ANNOTATIONS
# ---------------------------------------------------------
#
# Every target entity in these five scoring notes is annotated.

gold_development = pd.DataFrame([

    # M4-01
    make_reference(development_notes, "M4-01", "heroin", "SUBSTANCE"),
    make_reference(development_notes, "M4-01", "buprenorphine", "MEDICATION"),
    make_reference(development_notes, "M4-01", "opioid use disorder", "CONDITION"),

    # M4-03
    make_reference(development_notes, "M4-03", "methadone", "MEDICATION"),
    make_reference(development_notes, "M4-03", "OUD", "CONDITION"),
    make_reference(development_notes, "M4-03", "opioid", "SUBSTANCE"),

    # M4-04
    make_reference(development_notes, "M4-04", "Naloxone", "MEDICATION"),
    make_reference(development_notes, "M4-04", "4 mg", "DOSE"),
    make_reference(development_notes, "M4-04", "intranasal", "ROUTE"),

    # M4-09
    make_reference(development_notes, "M4-09", "bup", "MEDICATION"),

    # M4-10
    make_reference(development_notes, "M4-10", "fent", "SUBSTANCE")

], columns=[
    "note_id",
    "mention",
    "start",
    "end",
    "entity_type"
])


print("Development notes available for exploration:", len(development_notes))
print("Fully annotated development scoring notes:", len(development_scoring_notes))
print("Reference entities:", len(gold_development))
display(gold_development)


Development notes available for exploration: 11
Fully annotated development scoring notes: 5
Reference entities: 11


,note_id,mention,start,end,entity_type
0,M4-01,heroin,15,21,SUBSTANCE
1,M4-01,buprenorphine,37,50,MEDICATION
2,M4-01,opioid use disorder,55,74,CONDITION
3,M4-03,methadone,16,25,MEDICATION
4,M4-03,oud,30,33,CONDITION
5,M4-03,opioid,54,60,SUBSTANCE
6,M4-04,naloxone,0,8,MEDICATION
7,M4-04,4 mg,9,13,DOSE
8,M4-04,intranasal,14,24,ROUTE
9,M4-09,bup,23,26,MEDICATION


### Evaluate the Baseline

Now compare the baseline NER predictions with the human reference annotations to calculate precision, recall, and F1 and identify any missed or extra entities.

In [ ]:
def evaluate_entities(predicted_df, gold_df, scoring_note_ids):
    """
    Evaluate entity extraction using exact-span + entity-type matching.

    A prediction counts as correct only when `note_id`, `start`, `end`,
    and `entity_type` exactly match a reference annotation.

    Parameters
    ----------
    predicted_df : pandas.DataFrame
        NLP predictions containing `note_id`, `start`, `end`,
        and `entity_type`.

    gold_df : pandas.DataFrame
        Human-defined reference annotations with the same four fields.

    scoring_note_ids : list of str
        Fully annotated notes to score, including any with no target entities.
        Predictions outside this list are excluded, not counted as errors.

    Returns
    -------
    tuple
        metrics : dict
            True positives, false positives, false negatives,
            precision, recall, and F1.
        missed : list
            Reference entities the NLP system failed to predict.
        extra : list
            Predicted entities that were not in the reference set.
    """
    scoring_ids = set(scoring_note_ids)

    # Make sure all reference annotations belong to notes we intend to score.
    if not set(gold_df["note_id"]).issubset(scoring_ids):
        raise ValueError(
            "Reference annotations contain notes outside the scoring set."
        )

    # Ignore predictions from development notes that are not part
    # of the fully annotated scoring subset.
    predicted_df = predicted_df[
        predicted_df["note_id"].isin(scoring_ids)
    ]

    keys = ["note_id", "start", "end", "entity_type"]

    # Duplicate entity keys would make exact-match counting ambiguous.
    if predicted_df.duplicated(keys).any() or gold_df.duplicated(keys).any():
        raise ValueError(
            "Duplicate entity keys found; inspect annotations or predictions."
        )

    # Build an exact matching key for each predicted entity.
    predicted = {
        (row.note_id, int(row.start), int(row.end), row.entity_type)
        for row in predicted_df.itertuples()
    }

    # Build the same key for each reference entity.
    gold = {
        (row.note_id, int(row.start), int(row.end), row.entity_type)
        for row in gold_df.itertuples()
    }

    true_positive = len(predicted & gold)
    false_positive = len(predicted - gold)
    false_negative = len(gold - predicted)

    # Precision: Of what we predicted, how much was correct?
    precision = (
        true_positive / (true_positive + false_positive)
        if (true_positive + false_positive)
        else 0
    )

    # Recall: Of what we should have found, how much did we find?
    recall = (
        true_positive / (true_positive + false_negative)
        if (true_positive + false_negative)
        else 0
    )

    # F1: harmonic mean of precision and recall.
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0
    )

    missed = sorted(gold - predicted)
    extra = sorted(predicted - gold)

    metrics = {
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
    }

    return metrics, missed, extra


# ---------------------------------------------------------
# BASELINE DEVELOPMENT-SUBSET EVALUATION
# ---------------------------------------------------------

# Run the baseline pipeline only on the five fully annotated notes.
baseline_scoring_entities = run_pipeline(
    development_scoring_notes,
    term_patterns=BASE_TERM_PATTERNS,
    normalization_map=BASE_NORMALIZATION_MAP,
)

baseline_metrics, baseline_missed, baseline_extra = evaluate_entities(
    baseline_scoring_entities,
    gold_development,
    development_reference_ids
)


# ---------------------------------------------------------
# FORMAT RESULTS AS TABLES
# ---------------------------------------------------------

# Show the overall NER metrics as one summary row.
baseline_metrics_table = pd.DataFrame(
    [baseline_metrics],
    index=["baseline"]
)

# Convert missed entity keys back into a readable table.
missed_keys = pd.DataFrame(
    baseline_missed,
    columns=["note_id", "start", "end", "entity_type"]
)

baseline_missed_table = missed_keys.merge(
    gold_development[
        ["note_id", "mention", "start", "end", "entity_type"]
    ],
    on=["note_id", "start", "end", "entity_type"],
    how="left"
)[
    ["note_id", "mention", "start", "end", "entity_type"]
]

# Convert extra prediction keys back into a readable table.
extra_keys = pd.DataFrame(
    baseline_extra,
    columns=["note_id", "start", "end", "entity_type"]
)

baseline_extra_table = extra_keys.merge(
    baseline_scoring_entities[
        ["note_id", "mention", "start", "end", "entity_type"]
    ],
    on=["note_id", "start", "end", "entity_type"],
    how="left"
)[
    ["note_id", "mention", "start", "end", "entity_type"]
]


# ---------------------------------------------------------
# DISPLAY RESULTS
# ---------------------------------------------------------

print("BASELINE DEVELOPMENT-SUBSET NER RESULTS")
display(baseline_metrics_table)

print("MISSED ENTITIES (FALSE NEGATIVES)")
display(baseline_missed_table)

print("EXTRA PREDICTIONS (FALSE POSITIVES)")
display(baseline_extra_table)

BASELINE DEVELOPMENT-SUBSET NER RESULTS


,true_positive,false_positive,false_negative,precision,recall,f1
baseline,9,0,2,1.0,0.818,0.9


MISSED ENTITIES (FALSE NEGATIVES)


,note_id,mention,start,end,entity_type
0,M4-09,bup,23,26,MEDICATION
1,M4-10,fent,16,20,SUBSTANCE


EXTRA PREDICTIONS (FALSE POSITIVES)


,note_id,mention,start,end,entity_type


## Foundational Rule Improvement

Everyone continues here. The baseline metrics describe performance on the five annotated development notes.

1. Inspect the missed and extra entities below alongside their source notes.
2. Decide what caused each error: a missing recognition rule, a wrong span, or a wrong entity type?
3. Write the recognition and normalization changes you propose **before reading the following revision cell**. A normalization mapping alone cannot make NER find a missing mention.

Your proposed changes: *YOUR TEXT HERE*

The error tables join the evaluator's offset keys back to `gold_development` or predictions so you can read the actual mentions. An empty table means no errors of that kind were found.


In [ ]:
# Recover readable mentions and source notes for each error key.
entity_keys = ["note_id", "start", "end", "entity_type"]
missed_development = gold_development.merge(
    pd.DataFrame(baseline_missed, columns=entity_keys), on=entity_keys
).merge(development_scoring_notes[["note_id", "text"]], on="note_id")
extra_development = baseline_scoring_entities.merge(
    pd.DataFrame(baseline_extra, columns=entity_keys), on=entity_keys
)
print("MISSED REFERENCE ENTITIES")
display(missed_development[["note_id", "mention", "entity_type", "text"]])
print("EXTRA PREDICTIONS")
display(extra_development[["note_id", "mention", "start", "end", "entity_type"]])


MISSED REFERENCE ENTITIES


,note_id,mention,entity_type,text
0,M4-09,bup,MEDICATION,Patient reports taking bup as prescribed.
1,M4-10,fent,SUBSTANCE,Patient reports fent use was years ago.


EXTRA PREDICTIONS


,note_id,mention,start,end,entity_type


### Apply and check your development-based changes

Compare your proposed changes with the supplied revision below. It adds recognition rules and corresponding preferred terms for the missed shorthand mentions. Fresh dictionary copies keep the baseline unchanged.

Run the revised pipeline on all development notes for inspection, then score **only the same five annotated notes**. Compare baseline and revised precision, recall, and F1. Check `normalized_concept` separately: NER scores do not assess normalization. You may still revise rules at this development stage.


In [ ]:
# Apply the changes identified by inspecting development errors.
revised_term_patterns = copy.deepcopy(BASE_TERM_PATTERNS)
revised_normalization_map = copy.deepcopy(BASE_NORMALIZATION_MAP)
revised_term_patterns["MEDICATION"].append("bup")
revised_term_patterns["SUBSTANCE"].append("fent")
revised_normalization_map["bup"] = "buprenorphine"
revised_normalization_map["fent"] = "fentanyl"

revised_development_entities = run_pipeline(
    development_notes,
    term_patterns=revised_term_patterns,
    normalization_map=revised_normalization_map,
)
revised_scoring_entities = revised_development_entities[
    revised_development_entities["note_id"].isin(development_reference_ids)
].copy()
revised_development_metrics, revised_development_missed, revised_development_extra = evaluate_entities(
    revised_scoring_entities, gold_development, development_reference_ids
)
print("DEVELOPMENT SCORING SUBSET")
display(pd.DataFrame(
    [baseline_metrics, revised_development_metrics], index=["baseline", "revised"]
))
print("Remaining misses:", revised_development_missed or "None")
print("Extra predictions:", revised_development_extra or "None")
print("Newly recognized mentions and their normalization:")
display(revised_development_entities.loc[
    revised_development_entities["note_id"].isin(missed_development["note_id"]),
    ["note_id", "mention", "entity_type", "normalized_concept"]
])


DEVELOPMENT SCORING SUBSET


,true_positive,false_positive,false_negative,precision,recall,f1
baseline,9,0,2,1.0,0.818,0.9
revised,11,0,0,1.0,1.000,1.0


Remaining misses: None
Extra predictions: None
Newly recognized mentions and their normalization:


,note_id,mention,entity_type,normalized_concept
15,M4-09,bup,MEDICATION,buprenorphine
16,M4-10,fent,SUBSTANCE,fentanyl


### Confirm improvement and freeze the rules

With the supplied changes, both missed entities are now found: TP increases from **9 to 11**, FN falls from **2 to 0**, and FP remains **0**. Precision stays **1.000**, recall rises from **0.818 to 1.000**, and F1 rises from **0.900 to 1.000**. The new rows also show the intended normalization. These abbreviations are unambiguous in our synthetic examples, not necessarily in other clinical text.

The development revision is complete. Run the next cell to take independent copies named `final_term_patterns` and `final_normalization_map` and produce `final_development_entities` from those exact objects. From this point, keep the term lists, normalization map, preprocessing, and context functions unchanged. “Frozen” is a workflow commitment; the copies prevent later edits to the draft dictionaries from changing the finalized dictionaries.


In [ ]:
# Finalize the shared rule-based system before introducing any new evaluation notes.
final_term_patterns = copy.deepcopy(revised_term_patterns)
final_normalization_map = copy.deepcopy(revised_normalization_map)
final_development_entities = run_pipeline(
    development_notes,
    term_patterns=final_term_patterns,
    normalization_map=final_normalization_map,
)
final_development_scoring_entities = final_development_entities[
    final_development_entities["note_id"].isin(development_reference_ids)
].copy()
final_development_metrics, final_development_missed, final_development_extra = evaluate_entities(
    final_development_scoring_entities, gold_development, development_reference_ids
)
# The optional extension records its status separately from the core results.
pretrained_run_status = "not_run"
print("Rules finalized. Development F1:", final_development_metrics["f1"])


Rules finalized. Development F1: 1.0


### Final Check on Held-Out Notes

Development-based rule changes are complete and frozen. We now introduce **six new synthetic notes**, called the **held-out evaluation set**. Their text and reference answers were not used to choose the rule changes.

Run the next cells in order: reveal the notes, create their complete reference annotations, then evaluate the finalized pipeline **once**. This is a small independent check even though no model was trained. **Do not modify rules based on these results.** If results motivate future changes, those changes would require a new independent evaluation set.

Here too, reference annotations are answers attached to the notes, not another dataset.


In [ ]:
evaluation_notes = pd.DataFrame({
    "note_id": [
        "M4-E01",
        "M4-E02",
        "M4-E03",
        "M4-E04",
        "M4-E05",
        "M4-E06"
    ],

    "text": [
        "During follow-up, patient reports taking bup daily.",
        "At intake, patient reports fent use yesterday.",
        "Naloxone was discussed during overdose education. Patient denies heroin use.",
        "Patient is taking methadone for OUD.",
        "A 4 mg intranasal naloxone kit was offered.",
        "Years ago, patient reported fentanyl use."
    ]
})

evaluation_notes["split"] = "held_out_evaluation"

# Use exactly the same preprocessing as the development workflow.
evaluation_notes["text_processed"] = evaluation_notes["text"].map(preprocess_text)
print("HELD-OUT EVALUATION NOTES")
display(evaluation_notes[["note_id", "text"]])


HELD-OUT EVALUATION NOTES


,note_id,text
0,M4-E01,"During follow-up, patient reports taking bup daily."
1,M4-E02,"At intake, patient reports fent use yesterday."
2,M4-E03,Naloxone was discussed during overdose education. Patient denies heroin use.
3,M4-E04,Patient is taking methadone for OUD.
4,M4-E05,A 4 mg intranasal naloxone kit was offered.
5,M4-E06,"Years ago, patient reported fentanyl use."


### Reference answers for the held-out notes

Every target NER entity in all six notes is annotated below, following the same span and label policy as development. The `make_reference()` helper is unchanged. Inspect the reference table, then run the frozen pipeline; do not use these answers to add rules.


In [ ]:
# Complete human-defined NER answers for the held-out notes.
gold_evaluation = pd.DataFrame([

    # M4-E01
    make_reference(evaluation_notes, "M4-E01", "bup", "MEDICATION"),

    # M4-E02
    make_reference(evaluation_notes, "M4-E02", "fent", "SUBSTANCE"),

    # M4-E03
    make_reference(evaluation_notes, "M4-E03", "Naloxone", "MEDICATION"),
    make_reference(evaluation_notes, "M4-E03", "heroin", "SUBSTANCE"),

    # M4-E04
    make_reference(evaluation_notes, "M4-E04", "methadone", "MEDICATION"),
    make_reference(evaluation_notes, "M4-E04", "OUD", "CONDITION"),

    # M4-E05
    make_reference(evaluation_notes, "M4-E05", "4 mg", "DOSE"),
    make_reference(evaluation_notes, "M4-E05", "intranasal", "ROUTE"),
    make_reference(evaluation_notes, "M4-E05", "naloxone", "MEDICATION"),

    # M4-E06
    make_reference(evaluation_notes, "M4-E06", "fentanyl", "SUBSTANCE")

], columns=[
    "note_id",
    "mention",
    "start",
    "end",
    "entity_type"
])

print("Held-out reference entities:", len(gold_evaluation))
display(gold_evaluation)


Held-out reference entities: 10


,note_id,mention,start,end,entity_type
0,M4-E01,bup,41,44,MEDICATION
1,M4-E02,fent,27,31,SUBSTANCE
2,M4-E03,naloxone,0,8,MEDICATION
3,M4-E03,heroin,65,71,SUBSTANCE
4,M4-E04,methadone,18,27,MEDICATION
5,M4-E04,oud,32,35,CONDITION
6,M4-E05,4 mg,2,6,DOSE
7,M4-E05,intranasal,7,17,ROUTE
8,M4-E05,naloxone,18,26,MEDICATION
9,M4-E06,fentanyl,28,36,SUBSTANCE


In [ ]:
# One held-out evaluation of the finalized rule-based pipeline.
final_evaluation_entities = run_pipeline(
    evaluation_notes,
    term_patterns=final_term_patterns,
    normalization_map=final_normalization_map,
)
held_out_metrics, held_out_missed, held_out_extra = evaluate_entities(
    final_evaluation_entities, gold_evaluation, evaluation_notes["note_id"].tolist()
)
print("FINAL HELD-OUT NER RESULTS")
display(pd.DataFrame([held_out_metrics], index=["finalized rules"]))
print("Missed entities:", held_out_missed or "None")
print("Extra predictions:", held_out_extra or "None")

# Everyone exports this same rule-based table, with split labels retained.
final_entities = pd.concat(
    [final_development_entities, final_evaluation_entities], ignore_index=True
)


FINAL HELD-OUT NER RESULTS


,true_positive,false_positive,false_negative,precision,recall,f1
finalized rules,10,0,0,1.0,1.0,1.0


Missed entities: None
Extra predictions: None


### Interpret the final check

With the supplied finalized rules, the held-out result is **10 correct entities, no extras, and no misses**, giving precision, recall, and F1 of **1.000**. This shows that the revised rules also recognize the target entities in these new sentences. It does not establish real-world performance or accuracy of normalization and context.

Keep the held-out result separate from development results: development helped us choose rules; held-out data checked those fixed rules. We do not tune further or compare new revisions on this set.

### Note on Context Evaluation

Before completing the core activity, inspect one known context limitation from development. M4-11 is outside the five-note scoring subset. Its `fentanyl` span/type is correct on manual inspection, but its assertion is incorrectly `NEGATED`: “denies cravings” does not negate “reports fentanyl use.” Expected labels are `PRESENT`, `CURRENT` (including yesterday), `PATIENT`, and `SUBSTANCE_USE_OR_EXPOSURE`. Even an NER score including this note would not assess assertion.


In [ ]:
print("QUALITATIVE CONTEXT CHECK — not part of the NER score")
display(development_notes.loc[
    development_notes["note_id"] == "M4-11", ["note_id", "text"]
])
display(final_development_entities.loc[
    final_development_entities["note_id"] == "M4-11",
    ["mention", "entity_type", "assertion", "temporality", "experiencer", "use_context"]
])


QUALITATIVE CONTEXT CHECK — not part of the NER score


,note_id,text
10,M4-11,Patient denies cravings but reports fentanyl use yesterday.


,mention,entity_type,assertion,temporality,experiencer,use_context
17,fentanyl,SUBSTANCE,NEGATED,CURRENT,PATIENT,SUBSTANCE_USE_OR_EXPOSURE


### What Did We Actually Evaluate?

Our evaluation looked at different parts of the pipeline in different ways.

| Pipeline component | How we checked it | What we learned |
|---|---|---|
| **Entity recognition (NER)** | Precision, recall, and F1 against human reference annotations | After adding `bup` and `fent` to the NER term patterns, the finalized extractor correctly identified all 10 target entity spans and types in the small held-out set. |
| **Normalization** | Inspect the structured output | Extracted shorthand such as `bup` and `fent` is mapped to the preferred terms `buprenorphine` and `fentanyl`. |
| **Context interpretation** | Qualitative inspection of selected examples | Most simple examples behave as intended, but M4-11 shows that the rule-based assertion logic can misunderstand which part of a sentence a negation applies to. |

### Why does this matter?

An NER F1 score of **1.00** means that the system correctly found and classified the target entity spans in this small held-out set.

It does **not** mean that every part of the NLP pipeline is perfect.

For example, the system correctly finds `fentanyl` in M4-11, but incorrectly labels the mention as `NEGATED`. The error occurs after NER, during **context interpretation**.

> **Key takeaway:** Different stages of an NLP pipeline can fail independently. Evaluating entity recognition tells us whether the right entities were found, but normalization and context interpretation must also be checked when those outputs matter to the research question.

## Foundational Activity Complete

Answer briefly:

1. What did development error analysis reveal, and what recognition and normalization rules did you change? Explain the development metric changes.
2. Report held-out precision, recall, and F1 separately. Why is this a separate check without model training, and why must the rules stay frozen after seeing it?
3. Why is M4-11 a context error rather than an NER labeling error? What does this show about the limits of our scores?

Your answer:

- **Foundational learners:** Skip the optional Applied Extension and continue to **Shared FAIR / Provenance**.
- **Applied learners:** Continue below using the finalized Foundational pipeline. You will not repeat rule building or evaluation.


### Beyond This Rule-Based Baseline

The Foundational dictionaries and context cues are intentionally simple and transparent. **Pretrained biomedical/clinical NER models** learn patterns from annotated examples; **transformer- or LLM-based extraction approaches** can use surrounding language and, for some systems, instructions to structure information.

These methods still need task-specific evaluation. The optional Applied Extension now makes one small pretrained NER comparison with the system you have already finalized.


# Applied Extension: Comparing Rule-Based and Pretrained NER

This optional extension starts from `final_development_entities`, produced by the completed Foundational Activity. **Keep the finalized rule-based system unchanged.**


The rule-based extractor uses our term lists and dose/route patterns. A pretrained model instead learns patterns from annotated examples and can recognize language we did not list manually. This does not guarantee it will recognize shorthand correctly.

We use [`d4data/biomedical-ner-all`](https://huggingface.co/d4data/biomedical-ner-all), built on DistilBERT and trained on MACCROBAT biomedical case reports. It was not trained for this notebook's task. A **token** is a word, part of a word, or punctuation unit; the library's tokenizer prepares model input. Aggregation combines token predictions into labeled spans.

Compare only NER spans and labels on **four development notes**. The model does not supply our normalization or context fields. Its labels (such as `Medication`, `Disease_disorder`, and `Dosage`) differ from ours, so **do not calculate or compare model F1 using our rule-based reference labels** without first aligning both schemas and annotation policies.

In the model table, `mention`, `start`, and `end` locate the predicted span in processed text. `model_label` is the model's category. `confidence` is an aggregated model score between 0 and 1, **not a verified probability that the prediction is correct**. We use the same processed notes for both approaches; preprocessing choices can also affect model results.

The next cell installs the model dependencies and downloads public weights; it requires internet access and may take a few minutes. If loading fails, the notebook reports the failure and still permits rule-based export. Record that the model comparison was unavailable instead of inventing observations.


In [ ]:
# Install in this notebook's kernel; PyTorch is the model's execution backend.
%pip -q install transformers torch

PRETRAINED_MODEL_NAME = "d4data/biomedical-ner-all"
pretrained_ner = None
pretrained_run_status = "not_run"
pretrained_load_error = None
pretrained_revision = None

try:
    from transformers import pipeline

    # Average subword label scores within words, then group compatible labels.
    # Aggregation reduces fragmentation but does not guarantee correct boundaries.
    pretrained_ner = pipeline(
        "token-classification",
        model=PRETRAINED_MODEL_NAME,
        tokenizer=PRETRAINED_MODEL_NAME,
        aggregation_strategy="average",
        device=-1,  # CPU is sufficient for four short notes.
    )
    pretrained_revision = getattr(pretrained_ner.model.config, "_commit_hash", None)
    pretrained_run_status = "loaded"
    print(f"Loaded pretrained model: {PRETRAINED_MODEL_NAME}")
except (ImportError, OSError, RuntimeError, ValueError) as error:
    pretrained_run_status = "unavailable"
    pretrained_load_error = f"{type(error).__name__}: {error}"
    print("Model could not load:", pretrained_load_error)
    print("Check the installation/download message, then retry this cell in Colab.")


config.json:   0%|          | 0.00/5.00k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  266MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Loaded pretrained model: d4data/biomedical-ner-all


### Run the Pretrained Model and Compare Outputs

Now that the pretrained model is loaded, the next cell defines a small helper function that runs it on selected notes and organizes its predictions into a table.

We then:

1. use the **same four development notes** for both approaches,
2. retrieve the NER results from our finalized rule-based pipeline,
3. run the pretrained biomedical NER model on those notes, and
4. display the outputs so we can compare the entity spans and labels.

This comparison uses development notes only. The held-out evaluation has already been completed with the finalized rule-based system, so we do not use this exploratory comparison to change the rules or rescore the held-out set.

In [ ]:
def run_pretrained_ner(note_df, ner_pipeline, note_ids):
    """
    Apply a pretrained biomedical NER model to selected notes.

    Parameters
    ----------
    note_df : pandas.DataFrame
        Table containing `note_id` and `text_processed` columns.

    ner_pipeline : transformers.Pipeline
        Pretrained token-classification pipeline used for NER.

    note_ids : list of str
        IDs of the notes to analyze.

    Returns
    -------
    pandas.DataFrame
        One row per predicted entity, including:
        - note_id
        - mention
        - start and end character offsets
        - model_label
        - confidence

    Notes
    -----
    The model's entity labels come from its original biomedical
    training data. They should not be interpreted as equivalent to
    the custom entity categories used by our rule-based pipeline.

    The confidence score reflects the model's confidence in its own
    prediction. A high confidence score does not prove that the
    prediction is correct or appropriate for our research question.
    """
    rows = []

    # Keep only the selected development notes for this comparison.
    selected_notes = note_df[
        note_df["note_id"].isin(note_ids)
    ]

    # Run the pretrained model one note at a time.
    for _, note in selected_notes.iterrows():
        text = note["text_processed"]
        predictions = ner_pipeline(text)

        for prediction in predictions:
            start = int(prediction["start"])
            end = int(prediction["end"])

            # Recover the exact predicted span from the same text
            # that was supplied to the model.
            mention = text[start:end]

            rows.append({
                "note_id": note["note_id"],
                "mention": mention,
                "start": start,
                "end": end,
                "model_label": prediction.get(
                    "entity_group",
                    prediction.get("entity", "UNKNOWN")
                ),
                "confidence": round(float(prediction["score"]), 3),
            })

    columns = [
        "note_id",
        "mention",
        "start",
        "end",
        "model_label",
        "confidence",
    ]

    return pd.DataFrame(rows, columns=columns)


# ---------------------------------------------------------
# COMPARE BOTH APPROACHES ON THE SAME DEVELOPMENT NOTES
# ---------------------------------------------------------
#
# We use development notes only for this exploratory comparison.
# Held-out results were already obtained for the frozen rule-based pipeline.
# Do not use this exploratory comparison to revise those rules and rescore it.

model_comparison_ids = [
    "M4-01",
    "M4-04",
    "M4-10",
    "M4-11",
]

# Select results from the finalized Foundational rule-based pipeline.
rule_based_comparison = final_development_entities[
    final_development_entities["note_id"].isin(model_comparison_ids)
][[
    "note_id",
    "mention",
    "start",
    "end",
    "entity_type",
]].copy()

print("SOURCE DEVELOPMENT NOTES")
display(development_notes.loc[
    development_notes["note_id"].isin(model_comparison_ids), ["note_id", "text_processed"]
])
print("FINALIZED RULE-BASED NER")
display(rule_based_comparison.reset_index(drop=True))

pretrained_comparison = pd.DataFrame(columns=[
    "note_id", "mention", "start", "end", "model_label", "confidence"
])
if pretrained_ner is None:
    print("Pretrained comparison unavailable: rerun the model-loading cell in Colab.")
else:
    # Mark complete only after inference succeeds; a loaded model is not a result.
    pretrained_run_status = "inference_started"
    pretrained_comparison = run_pretrained_ner(
        development_notes, pretrained_ner, model_comparison_ids
    )
    pretrained_run_status = "completed"
    print("PRETRAINED BIOMEDICAL NER")
    display(pretrained_comparison)


SOURCE DEVELOPMENT NOTES


,note_id,text_processed
0,M4-01,patient denies heroin use. continues buprenorphine for opioid use disorder.
3,M4-04,naloxone 4 mg intranasal kit offered for overdose education.
9,M4-10,patient reports fent use was years ago.
10,M4-11,patient denies cravings but reports fentanyl use yesterday.


FINALIZED RULE-BASED NER


,note_id,mention,start,end,entity_type
0,M4-01,heroin,15,21,SUBSTANCE
1,M4-01,buprenorphine,37,50,MEDICATION
2,M4-01,opioid use disorder,55,74,CONDITION
3,M4-04,naloxone,0,8,MEDICATION
4,M4-04,4 mg,9,13,DOSE
5,M4-04,intranasal,14,24,ROUTE
6,M4-10,fent,16,20,SUBSTANCE
7,M4-11,fentanyl,36,44,SUBSTANCE


PRETRAINED BIOMEDICAL NER


,note_id,mention,start,end,model_label,confidence
0,M4-01,heroin,15,21,Medication,0.999
1,M4-01,buprenorphine,37,50,Medication,0.991
2,M4-04,naloxone,0,8,Medication,0.509
3,M4-04,4 mg,9,13,Dosage,0.994
4,M4-04,intranasal,14,24,Administration,0.478
5,M4-04,overdose,41,49,Administration,0.696
6,M4-10,fent,16,20,Activity,0.456
7,M4-11,cravings,15,23,Sign_symptom,0.500
8,M4-11,fentanyl,36,44,Medication,0.663


### Interpret the comparison

Use the output from **your run**, rather than assuming a particular prediction will appear. Separate three kinds of differences:

- **Different scope or labels:** `Administration` may correspond to our `ROUTE`; `Sign_symptom` could cover `cravings`, which is outside our target inventory. Extra model categories are not automatically errors.
- **Potential errors:** Look for missed entities, implausible labels, or incorrect span boundaries. If `fent` is labeled `Activity`, for example, inspect its sentence before interpreting it.
- **Missing functionality:** A model may find `fentanyl`, but NER alone does not decide whether its use is affirmed or denied. A high confidence value cannot resolve this limitation.

This is an exploratory comparison, not evidence that either approach is generally superior.


### What Does NER Still Leave Unresolved?

The comparison above focused on **entity recognition**: which text spans each approach identified and how they labeled them.

But our rule-based workflow does more than NER. After an entity is identified, it also assigns:

- a normalized concept
- assertion
- temporality
- experiencer
- use context

A pretrained NER model does not automatically provide these additional interpretations.

The next output returns to three **development notes** to show why these context fields matter. These notes illustrate different kinds of contextual information and one limitation of our simple context rules.

We use development notes here because this is a **qualitative review**, not another held-out evaluation. The finalized rules remain frozen.

In [ ]:
# Context review stays qualitative and uses development notes only.
context_review_ids = ["M4-03", "M4-10", "M4-11"]
display(development_notes.loc[
    development_notes["note_id"].isin(context_review_ids), ["note_id", "text"]
])
display(final_development_entities.loc[
    final_development_entities["note_id"].isin(context_review_ids),
    ["note_id", "mention", "normalized_concept", "assertion", "temporality", "experiencer", "use_context"]
])


,note_id,text
2,M4-03,Father receives methadone for OUD.\tPatient reports no opioid use.
9,M4-10,Patient reports fent use was years ago.
10,M4-11,Patient denies cravings but reports fentanyl use yesterday.


,note_id,mention,normalized_concept,assertion,temporality,experiencer,use_context
5,M4-03,methadone,methadone,PRESENT,CURRENT,FAMILY,TREATMENT
6,M4-03,oud,opioid use disorder,PRESENT,CURRENT,FAMILY,CONDITION
7,M4-03,opioid,opioid,NEGATED,CURRENT,PATIENT,SUBSTANCE_USE_OR_EXPOSURE
16,M4-10,fent,fentanyl,PRESENT,HISTORICAL,PATIENT,SUBSTANCE_USE_OR_EXPOSURE
17,M4-11,fentanyl,fentanyl,NEGATED,CURRENT,PATIENT,SUBSTANCE_USE_OR_EXPOSURE


### What Should You Notice?

These examples show that entity recognition and context interpretation are separate tasks:

- **M4-03:** the pipeline distinguishes mentions referring to a family member from those referring to the patient.
- **M4-10:** the pipeline recognizes that the substance-use mention is historical.
- **M4-11:** the entity `fentanyl` is found correctly, but its assertion is incorrectly labeled `NEGATED`. The phrase “denies cravings” does not actually negate “reports fentanyl use.”

This last example shows why even a transparent rule-based system can fail when language becomes more complex. The NER step can be correct while a later context-interpretation step is wrong.

### Applied Error Analysis and Reflection

Build on your Foundational response in one short paragraph.

1. Give one meaningful similarity or difference between the pretrained and rule-based NER outputs. Distinguish a difference in **label definitions/task scope** from a likely model error.

2. Give one strength and one limitation of each approach for this task. Why does a high model-confidence score not guarantee correctness or task fit?

3. Using one of the context examples above, explain one important piece of information that **NER alone does not provide**.

Use observations from your actual model output. If the pretrained model failed to load, record the error rather than inventing results.


## Shared FAIR / Provenance

Everyone completes this section. **FAIR** means Findable, Accessible, Interoperable, and Reusable. **Provenance** records where a result came from and how it was produced.

Export the finalized **rule-based** entity table as CSV and its documentation as JSON. The CSV contains all development and held-out predictions, identified by `data_split`; only the documented scoring notes contribute to metrics. The metadata records `pipeline_version`, rules, source notes, offset conventions, separate evaluation results, and limitations. Retain this notebook for the context-rule implementation and your responses.

If you completed Applied, the metadata also records the model identifier, resolved revision, and run status. The comparison table stays in the notebook; it is not mixed into the primary rule-based CSV. Foundational export does not depend on executing Applied cells.

Run the next cell. You do not need to edit the code; simply run the cell. In Colab, open **Files → outputs** and download the CSV and JSON. These documentation practices support FAIR; they do not establish full FAIR compliance or imply that real clinical data should be public.


In [ ]:
# ---------------------------------------------------------
# EXPORT FINAL RULE-BASED RESULTS AND PROVENANCE
# ---------------------------------------------------------
#
# No edits are required in this cell.
# It exports the finalized rule-based results for everyone
# and records Applied model information only if that section was run.

from datetime import datetime, timezone

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "module4_structured_entities.csv"
metadata_path = OUTPUT_DIR / "module4_output_metadata.json"


# ---------------------------------------------------------
# EXPORT FINAL STRUCTURED ENTITY TABLE
# ---------------------------------------------------------

final_entities.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)


# ---------------------------------------------------------
# COLLECT PROVENANCE INFORMATION
# ---------------------------------------------------------

# Keep the source notes so exported character spans can be traced
# back to the processed text from which they were extracted.
source_notes = pd.concat(
    [development_notes, evaluation_notes],
    ignore_index=True
)

# The Applied Extension is optional. These values allow the export
# to work even when those cells were never run.
pretrained_status_for_export = globals().get(
    "pretrained_run_status",
    "not_run"
)

model_was_attempted = pretrained_status_for_export != "not_run"


metadata = {
    "module": "WISER Module 4: NLP Semantics",
    "pipeline_version": PIPELINE_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "data_type": "fully synthetic teaching notes",
    "contains_phi": False,

    "structured_output_file": output_path.name,
    "structured_output_rows": len(final_entities),
    "structured_output_columns": list(final_entities.columns),

    "source_notes": source_notes.to_dict(orient="records"),

    "offset_convention": (
        "Zero-based start inclusive, end exclusive, in text_processed; "
        "raw offsets align for these ASCII notes."
    ),

    "preprocessing": (
        "Lowercase; replace each newline/tab with one space."
    ),

    "rules": {
        "term_patterns": final_term_patterns,
        "normalization_map": final_normalization_map,
        "dose_pattern": DOSE_PATTERN,
        "route_terms": ROUTE_TERMS,
        "context_implementation": (
            "interpret_context and sentence_for_span "
            "in the accompanying notebook."
        ),
    },

    "evaluation": {
        "matching_rule": (
            "Exact note_id, start, end, entity_type; "
            "entity counts pooled within each scoring set."
        ),

        "zero_denominator_policy": (
            "Report 0; metrics rounded to 3 decimal places."
        ),

        "development_scoring_note_ids": development_reference_ids,

        "held_out_scoring_note_ids": evaluation_notes[
            "note_id"
        ].tolist(),

        "baseline_development_metrics": baseline_metrics,
        "final_development_metrics": final_development_metrics,
        "final_held_out_metrics": held_out_metrics,

        "protocol": (
            "Choose changes on development, freeze rules, then evaluate "
            "finalized rules once on held-out notes; no model trained."
        ),

        "not_scored": [
            "normalization",
            "assertion",
            "temporality",
            "experiencer",
            "use_context",
        ],
    },

    "pretrained_model_comparison": {
        "status": pretrained_status_for_export,

        "model_name": (
            globals().get("PRETRAINED_MODEL_NAME")
            if model_was_attempted
            else None
        ),

        "resolved_revision": (
            globals().get("pretrained_revision")
            if model_was_attempted
            else None
        ),

        "note_ids": (
            globals().get("model_comparison_ids", [])
            if model_was_attempted
            else []
        ),

        "load_error": (
            globals().get("pretrained_load_error")
            if model_was_attempted
            else None
        ),

        "aggregation_strategy": (
            "average"
            if model_was_attempted
            else None
        ),

        "comparison_type": (
            "Qualitative only; schemas are not aligned for F1 comparison."
        ),

        "output_location": (
            "Comparison table in the notebook; "
            "excluded from rule-based CSV."
        ),
    },

    "concept_linking_performed": False,

    "limitations": [
        (
            "Tiny synthetic teaching sets; not an estimate of "
            "real-world clinical performance."
        ),
        (
            "M4-11 retains a known negation-scope error; "
            "context fields are not scored."
        ),
        (
            "Context defaults are heuristics; normalization and "
            "abbreviation choices are task-specific."
        ),
        (
            "No medication-attribute relation extraction or "
            "RxNorm identifiers assigned."
        ),
    ],

    "fair_notes": {
        "findable": (
            "Descriptive filenames and dataset-local note IDs."
        ),
        "accessible": (
            "Local course outputs; no public access is implied."
        ),
        "interoperable": (
            "UTF-8 CSV/JSON with consistent custom labels, "
            "not a standard ontology."
        ),
        "reusable": (
            "Keep CSV, metadata, and notebook together for rules, "
            "source evidence, and limitations."
        ),
    },
}


# ---------------------------------------------------------
# SAVE METADATA
# ---------------------------------------------------------

metadata_path.write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ---------------------------------------------------------
# CONFIRM EXPORT
# ---------------------------------------------------------

print("Saved:", output_path)
print("Saved:", metadata_path)
print("Rule-based rows:", len(final_entities))
print("Pipeline version:", PIPELINE_VERSION)
print(
    "Optional pretrained comparison status:",
    pretrained_status_for_export
)

Saved: outputs/module4_structured_entities.csv
Saved: outputs/module4_output_metadata.json
Rule-based rows: 28
Pipeline version: WISER-M4-teaching-v5
Optional pretrained comparison status: completed


## Final Knowledge Check

Check each box when you can explain it; revisit the relevant step if needed.

- [ ] I can trace raw text through preprocessing, NER, normalization, context, and structured output.
- [ ] I can distinguish finding a mention from evidence of current patient use.
- [ ] I can explain why reference annotations are answers for scored notes, not a third dataset.
- [ ] I can explain exact-span/type precision, recall, and F1 and why only fully annotated notes are scored.
- [ ] I can explain how development errors guide rule changes, when rules become frozen, and why held-out evaluation comes afterward.
- [ ] I understand that correct entity extraction does not establish correct context interpretation.
- [ ] **Applied:** I can interpret label differences and model confidence without directly comparing unaligned F1 scores.
- [ ] I can use source evidence and provenance to audit an exported row and explain one limitation.


## What to Submit

Everyone submits:

1. Completed notebook with all required cells run, outputs visible, and your responses added.
2. `module4_structured_entities.csv`
3. `module4_output_metadata.json`

**Applied learners also include** the pretrained comparison output and **Applied response** in the same notebook. If model loading was blocked, document the error and incomplete comparison explicitly. Foundational learners do not need to execute or answer Applied cells.

Report development and held-out precision, recall, and F1 separately. No separate final reflection or model CSV is required. This is a synthetic teaching workflow, not a clinically validated system.


## Optional References

- **RxNorm (U.S. National Library of Medicine):** https://www.nlm.nih.gov/research/umls/rxnorm/
- **FAIR Principles:** https://www.go-fair.org/fair-principles/
- **Pretrained biomedical NER model used in the Applied Extension:** https://huggingface.co/d4data/biomedical-ner-all

These references are optional for completing the notebook.
